In [ ]:
"""
Constraint-Driven Warm-Freeze (CDWF) - CIFAR-100

Demonstrates CDWF on CIFAR-100 with user-specified efficiency constraints.
"""

import os
import gc
import math
import random
import time
import copy

from tqdm import tqdm
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import OneCycleLR

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


# ================================================================
# GPU POWER MONITORING
# ================================================================
try:
    import pynvml

    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print("GPU power monitoring enabled")
except Exception:
    NVML_AVAILABLE = False
    _nvml_handle = None
    print("GPU power monitoring unavailable")


def read_gpu_power_w():
    if not NVML_AVAILABLE:
        return 0.0
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_nvml_handle) / 1000.0
    except Exception:
        return 0.0


# ================================================================
# CONFIGURATION
# ================================================================
EXPERIMENT_NAME = "cifar100_resnet50_cdwf"
SEED = 42

# Model
MODEL_DEPTH = 50
USE_PRETRAINED = True

# Training
BATCH_SIZE = 64
NUM_WORKERS = 2
MAX_LR = 3e-4
WEIGHT_DECAY = 1e-2
CLIP_NORM = 1.0

# Constraint-driven configuration
EPOCHS_FULLFT = 10

# Constraint settings
CONSTRAINT_TYPE = "params"  # Options: "speedup", "params", "energy", "balanced"
TARGET_SPEEDUP = 2.0
TARGET_PARAMS_PCT = 0.05
TARGET_ENERGY_REDUCTION = 2.5

MAX_TRAINABLE_FRACTION = 0.05
CONSTRAINT_TOLERANCE = 0.02

# Warm-start and fine-tuning
EPOCHS_WARMSTART_BRIEF = 3
EPOCHS_FINETUNE = 7

# LoRA options
LORA_RANKS = [1, 2, 4, 8, 16]
LORA_ALPHA = 1.0
BEST_RANK = 16

# Data
NUM_CLASSES = 100
DATA_DIR = "./data"
FULLFT_CSV = "cifar100_resnet50_fullft_results.csv"
RESULT_CSV = "cifar100_resnet50_cdwf_results.csv"
CHECKPOINT_PATH = "checkpoints/2fullft_resnet50_cifar100.pth"

print(f"\n{'=' * 70}")
print("  CDWF - CIFAR-100")
print(f"{'=' * 70}")
print(f"Model: ResNet-{MODEL_DEPTH}")
print(f"Constraint: {CONSTRAINT_TYPE.upper()}")
print(f"Target speedup: {TARGET_SPEEDUP:.1f}x")
print(f"Classes: {NUM_CLASSES}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print(f"{'=' * 70}\n")


# ================================================================
# REPRODUCIBILITY
# ================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")


# ================================================================
# DATA LOADING
# ================================================================
print("Loading CIFAR-100...")

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset_full = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform_train,
)
test_dataset = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform_test,
)

train_size = int(0.80 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Train: {train_size}, Val: {val_size}, Test: {len(test_dataset)}\n")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ================================================================
# LORA IMPLEMENTATION
# ================================================================
class LoRAConv2d(nn.Module):
    def __init__(self, base_conv, rank=1, alpha=1.0):
        super().__init__()
        self.base = base_conv
        for param in self.base.parameters():
            param.requires_grad = False

        self.rank = rank
        self.alpha = alpha

        in_ch = base_conv.in_channels
        out_ch = base_conv.out_channels
        kernel_size = base_conv.kernel_size[0]
        stride = base_conv.stride
        padding = base_conv.padding

        self.lora_A = nn.Conv2d(
            in_ch,
            rank,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.lora_B = nn.Conv2d(
            rank,
            out_ch,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )

        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.lora_B(self.lora_A(x))
        return base_out + (self.alpha / self.rank) * lora_out


# ================================================================
# MODEL UTILITIES
# ================================================================
def get_resnet_cifar100(depth=50, pretrained=True):
    model_fns = {
        18: models.resnet18,
        50: models.resnet50,
        101: models.resnet101,
    }
    model = model_fns[depth](pretrained=pretrained)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total - trainable


def get_block_dict(model):
    blocks = {}
    for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name)
        for i, block in enumerate(layer):
            blocks[f"{layer_name}_{i}"] = block
    return blocks


def get_block_param_count(block):
    return sum(p.numel() for p in block.parameters())


# ================================================================
# TRAINING FUNCTIONS
# ================================================================
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, scheduler, device, power_log=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in tqdm(loader, desc="Train", leave=False):
        if power_log is not None:
            now = time.time()
            power_log["energy_J"] += read_gpu_power_w() * (now - power_log["last_t"])
            power_log["last_t"] = now

        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        if CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)

        optimizer.step()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in tqdm(loader, desc="Eval", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


# ================================================================
# PREDICTION AND OPTIMIZATION
# ================================================================
def compute_block_features(model, val_loader, device, num_batches=10):
    print("\n[Feature Extraction] Analyzing block characteristics...")

    model.eval()
    blocks = get_block_dict(model)

    features = {
        name: {
            "grad_mean": 0.0,
            "grad_std": 0.0,
            "grad_samples": [],
            "param_count": get_block_param_count(block),
            "layer_depth": int(name.split("_")[0][-1]),
        }
        for name, block in blocks.items()
    }

    for batch_idx, (inputs, targets) in enumerate(val_loader):
        if batch_idx >= num_batches:
            break

        inputs, targets = inputs.to(device), targets.to(device)
        model.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        for name, block in blocks.items():
            grad_norm = 0.0
            for param in block.parameters():
                if param.grad is not None:
                    grad_norm += param.grad.norm().item() ** 2
            grad_norm = math.sqrt(grad_norm)
            features[name]["grad_samples"].append(grad_norm)

    for name in features:
        samples = features[name]["grad_samples"]
        features[name]["grad_mean"] = np.mean(samples)
        features[name]["grad_std"] = np.std(samples)
        del features[name]["grad_samples"]

    return features


def build_predictor(block_features, warmstart_acc, fullft_acc):
    print("\n[Predictive Model] Building accuracy predictor...")

    total_grad = sum(f["grad_mean"] for f in block_features.values())

    block_importance = {}
    for name, feats in block_features.items():
        block_importance[name] = feats["grad_mean"] / total_grad

    max_gain = fullft_acc - warmstart_acc

    print(f"  Warm-start acc: {warmstart_acc:.2f}%")
    print(f"  Full_ft acc: {fullft_acc:.2f}%")
    print(f"  Maximum gain: {max_gain:.2f}%")

    return block_importance, max_gain, warmstart_acc


def predict_accuracy(block_importance, max_gain, warmstart_acc, kept_blocks, lora_blocks, lora_rank):
    kept_gain = sum(block_importance[b] for b in kept_blocks)
    lora_efficiency = min(0.5, lora_rank / 8.0)
    lora_gain = sum(block_importance[b] * lora_efficiency for b in lora_blocks)

    total_gain_fraction = kept_gain + lora_gain

    if max_gain > 10.0:
        epoch_discount = 0.4
    elif max_gain > 8.0:
        epoch_discount = 0.7
    else:
        epoch_discount = 0.9

    predicted_acc = warmstart_acc + (total_gain_fraction * max_gain * epoch_discount)
    return predicted_acc


def estimate_config_cost(model, kept_blocks, lora_blocks, lora_rank, warmstart_epochs, finetune_epochs, fullft_time):
    """
    Estimate training time using fixed overhead, trainable fraction,
    and LoRA overhead.
    """
    blocks = get_block_dict(model)
    total_params, _, _ = count_params(model)

    trainable = 0
    trainable += sum(p.numel() for p in model.fc.parameters())

    for name in kept_blocks:
        trainable += get_block_param_count(blocks[name])

    for name in lora_blocks:
        block = blocks[name]
        if hasattr(block, "conv2"):
            conv = block.conv2
            in_ch = conv.in_channels
            out_ch = conv.out_channels
            kernel_size = conv.kernel_size[0]
            trainable += lora_rank * (in_ch * kernel_size * kernel_size + out_ch)

    trainable_fraction = trainable / total_params

    fixed_fraction = 0.55
    variable_fraction = 0.45

    warmstart_time = (warmstart_epochs / EPOCHS_FULLFT) * fullft_time

    finetune_base = (finetune_epochs / EPOCHS_FULLFT) * fullft_time
    finetune_time = finetune_base * (
        fixed_fraction + variable_fraction * trainable_fraction
    )

    n_lora_blocks = len(lora_blocks)

    if lora_rank == 1:
        lora_overhead_per_block = 0.15
    elif lora_rank == 2:
        lora_overhead_per_block = 0.10
    elif lora_rank == 4:
        lora_overhead_per_block = 0.05
    else:
        lora_overhead_per_block = 0.02

    if n_lora_blocks > 0:
        total_lora_overhead = 1.0 + (lora_overhead_per_block * n_lora_blocks)
        finetune_time *= total_lora_overhead

    apply_overhead = 3.0

    total_time = warmstart_time + finetune_time + apply_overhead
    speedup = fullft_time / total_time

    return {
        "trainable_params": trainable,
        "trainable_fraction": trainable_fraction,
        "estimated_time": total_time,
        "speedup": speedup,
    }


def solve_optimal_architecture(
    block_features,
    block_importance,
    max_gain,
    warmstart_acc,
    model,
    fullft_acc,
    fullft_time,
    constraint_type,
    target_value,
):
    print("\n[Optimization] Solving for optimal architecture...")
    print(f"  Constraint: {constraint_type} = {target_value}")

    block_names = list(block_features.keys())
    n_blocks = len(block_names)

    sorted_blocks = sorted(
        block_names,
        key=lambda b: block_importance[b],
        reverse=True,
    )

    best_config = None
    best_score = -float("inf")

    for n_keep in range(0, n_blocks + 1):
        kept = sorted_blocks[:n_keep]
        frozen = sorted_blocks[n_keep:]

        for lora_rank in LORA_RANKS:
            pred_acc = predict_accuracy(
                block_importance,
                max_gain,
                warmstart_acc,
                kept,
                frozen,
                lora_rank,
            )

            cost = estimate_config_cost(
                model,
                kept,
                frozen,
                lora_rank,
                EPOCHS_WARMSTART_BRIEF,
                EPOCHS_FINETUNE,
                fullft_time,
            )

            constraint_met = False
            if constraint_type == "speedup":
                constraint_met = (
                    cost["speedup"] >= target_value
                    and cost["trainable_fraction"] <= MAX_TRAINABLE_FRACTION
                )
            elif constraint_type == "params":
                constraint_met = cost["trainable_fraction"] <= target_value
            elif constraint_type == "energy":
                constraint_met = (
                    cost["speedup"] >= target_value
                    and cost["trainable_fraction"] <= MAX_TRAINABLE_FRACTION
                )
            else:
                constraint_met = (
                    cost["speedup"] >= 2.0
                    and cost["trainable_fraction"] <= 0.05
                )

            if not constraint_met:
                continue

            score = pred_acc

            if score > best_score:
                best_score = score
                best_config = {
                    "kept_blocks": kept.copy(),
                    "lora_blocks": frozen.copy(),
                    "lora_rank": lora_rank,
                    "predicted_acc": pred_acc,
                    "trainable_params": cost["trainable_params"],
                    "trainable_fraction": cost["trainable_fraction"],
                    "estimated_speedup": cost["speedup"],
                    "estimated_time": cost["estimated_time"],
                }

    if best_config is None:
        print("  No configuration meets the constraint. Using fallback...")

        fallback_strategies = [
            (1, BEST_RANK),
            (0, BEST_RANK),
        ]

        tolerance_levels = [
            CONSTRAINT_TOLERANCE,
            CONSTRAINT_TOLERANCE * 2,
            CONSTRAINT_TOLERANCE * 5,
        ]

        for tolerance_multiplier, tolerance in enumerate(tolerance_levels, 1):
            print(f"\n  Attempt {tolerance_multiplier}: tolerance = {tolerance * 100:.1f}%")
            print(
                f"    Param limit: {MAX_TRAINABLE_FRACTION * (1 + tolerance) * 100:.2f}%"
            )

            for i, (n_kept, lora_rank) in enumerate(fallback_strategies):
                if tolerance_multiplier == 1:
                    print(
                        f"    Fallback {i + 1}/{len(fallback_strategies)}: "
                        f"{n_kept} kept, rank={lora_rank}"
                    )

                if n_kept > 0:
                    kept = sorted_blocks[:n_kept]
                    frozen = sorted_blocks[n_kept:]
                else:
                    kept = []
                    frozen = sorted_blocks

                cost = estimate_config_cost(
                    model,
                    kept,
                    frozen,
                    lora_rank,
                    EPOCHS_WARMSTART_BRIEF,
                    EPOCHS_FINETUNE,
                    fullft_time,
                )

                if tolerance_multiplier == 1:
                    print(
                        f"      Speedup: {cost['speedup']:.2f}x, "
                        f"Params: {cost['trainable_fraction'] * 100:.3f}%"
                    )

                speedup_ok = cost["speedup"] >= target_value
                params_ok = (
                    cost["trainable_fraction"]
                    <= MAX_TRAINABLE_FRACTION * (1 + tolerance)
                )

                if speedup_ok and params_ok:
                    if tolerance > CONSTRAINT_TOLERANCE:
                        print(
                            f"  Used relaxed constraint: {tolerance * 100:.1f}% tolerance "
                            f"(original: {CONSTRAINT_TOLERANCE * 100:.1f}%)"
                        )

                    print(f"  Configuration found: {n_kept} kept, rank={lora_rank}")
                    print(
                        f"    Speedup: {cost['speedup']:.2f}x "
                        f"(target: {target_value:.2f}x)"
                    )
                    print(
                        f"    Params: {cost['trainable_fraction'] * 100:.3f}% "
                        f"(limit: {MAX_TRAINABLE_FRACTION * 100:.2f}%, "
                        f"effective: {MAX_TRAINABLE_FRACTION * (1 + tolerance) * 100:.2f}%)"
                    )

                    pred_acc = predict_accuracy(
                        block_importance,
                        max_gain,
                        warmstart_acc,
                        kept,
                        frozen,
                        lora_rank,
                    )

                    best_config = {
                        "kept_blocks": kept,
                        "lora_blocks": frozen,
                        "lora_rank": lora_rank,
                        "predicted_acc": pred_acc,
                        "trainable_params": cost["trainable_params"],
                        "trainable_fraction": cost["trainable_fraction"],
                        "estimated_speedup": cost["speedup"],
                        "estimated_time": cost["estimated_time"],
                        "used_tolerance": tolerance,
                    }
                    break

            if best_config is not None:
                break

        if best_config is None:
            print("\n  No configuration found even with relaxed constraints.")
            raise ValueError(
                f"Cannot find configuration satisfying constraints:\n"
                f"  Speedup target: {target_value:.2f}x\n"
                f"  Max params: {MAX_TRAINABLE_FRACTION * 100:.2f}%\n"
                f"  Tried tolerances up to: {tolerance_levels[-1] * 100:.1f}%\n\n"
                f"Most aggressive config (0 kept, rank=1):\n"
                f"  Speedup: {cost['speedup']:.2f}x\n"
                f"  Params: {cost['trainable_fraction'] * 100:.3f}%\n\n"
                f"Recommendation: Set MAX_TRAINABLE_FRACTION = {cost['trainable_fraction']:.4f}"
            )

    print("\n[Optimal Configuration]")
    print(f"  Keep trainable: {len(best_config['kept_blocks'])} blocks")
    print(
        f"  Freeze + LoRA rank-{best_config['lora_rank']}: "
        f"{len(best_config['lora_blocks'])} blocks"
    )
    print(f"  Predicted: {best_config['predicted_acc']:.2f}%")
    print(f"  Estimated speedup: {best_config['estimated_speedup']:.2f}x")

    return best_config


# ================================================================
# LOAD FULL_FT BASELINE
# ================================================================
print("=" * 70)
print("  LOADING CIFAR-100 FULL_FT BASELINE")
print("=" * 70 + "\n")

if os.path.exists(FULLFT_CSV):
    print(f"Loading baseline from: {FULLFT_CSV}")
    fullft_df = pd.read_csv(FULLFT_CSV)
    fullft_test_acc = fullft_df["test_acc"].iloc[0]
    fullft_val_acc = fullft_df["val_acc"].iloc[0]
    fullft_time = fullft_df["training_time_s"].iloc[0]
    fullft_energy = fullft_df["energy_J"].iloc[0]

    print("Loaded Full_FT baseline from CSV")
    print(f"  Val: {fullft_val_acc:.2f}%")
    print(f"  Test: {fullft_test_acc:.2f}%")
    print(f"  Time: {fullft_time:.1f}s")
    print(f"  Energy: {fullft_energy / 1000:.2f}kJ\n")

    if os.path.exists(CHECKPOINT_PATH):
        checkpoint = torch.load(CHECKPOINT_PATH)
        print(f"Checkpoint available: {CHECKPOINT_PATH}\n")
    else:
        print(f"Checkpoint not found: {CHECKPOINT_PATH}\n")

elif os.path.exists(CHECKPOINT_PATH):
    print(f"Loading baseline from checkpoint: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH)
    fullft_val_acc = checkpoint.get("best_val_acc", 70.0)
    fullft_test_acc = checkpoint.get("fullft_test_acc", 70.0)
    fullft_time = checkpoint.get("train_time_s", 500.0)
    fullft_energy = checkpoint.get("energy_J", 60000.0)

    print("Loaded Full_FT baseline from checkpoint")
    print(f"  Val: {fullft_val_acc:.2f}%")
    print(f"  Test: {fullft_test_acc:.2f}%")
    print(f"  Time: {fullft_time:.1f}s")
    print(f"  Energy: {fullft_energy / 1000:.2f}kJ\n")
else:
    print("Error: no baseline found.")
    print("Run the CIFAR-100 full fine-tuning script first.\n")
    raise SystemExit(1)


# ================================================================
# PHASE 2: CONSTRAINT-DRIVEN WARM-FREEZE
# ================================================================
print("=" * 70)
print("  PHASE 2: CONSTRAINT-DRIVEN WARM-FREEZE")
print("=" * 70 + "\n")

torch.cuda.empty_cache()
gc.collect()

cdwf_total_start = time.time()


# ================================================================
# STEP 1: BRIEF WARM-START
# ================================================================
print(f"[STEP 1] Brief Warm-Start ({EPOCHS_WARMSTART_BRIEF} epochs)")
print("-" * 70)

warmstart_start = time.time()
power_log = {"energy_J": 0.0, "last_t": warmstart_start}

model = get_resnet_cifar100(MODEL_DEPTH, USE_PRETRAINED).to(device)

optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS_WARMSTART_BRIEF,
    pct_start=0.3,
    div_factor=10,
    final_div_factor=100,
)

for epoch in range(1, EPOCHS_WARMSTART_BRIEF + 1):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device, power_log=power_log
    )
    val_loss, val_acc = evaluate(model, val_loader, device)
    print(
        f"  Epoch {epoch}/{EPOCHS_WARMSTART_BRIEF}: "
        f"train={train_acc:.2f}% val={val_acc:.2f}%"
    )

warmstart_time = time.time() - warmstart_start
warmstart_energy = power_log["energy_J"]
warmstart_acc = val_acc

print(f"\nWarm-start: {warmstart_acc:.2f}%")
print(f"  Time: {warmstart_time:.1f}s")
print(f"  Energy: {warmstart_energy / 1000:.2f}kJ\n")


# ================================================================
# STEP 2: FEATURE EXTRACTION AND PREDICTION
# ================================================================
print("[STEP 2] Feature Extraction and Predictive Modeling")
print("-" * 70)

feature_start = time.time()

block_features = compute_block_features(model, val_loader, device, num_batches=10)
block_importance, max_gain, _ = build_predictor(
    block_features,
    warmstart_acc,
    fullft_val_acc,
)

feature_time = time.time() - feature_start
print(f"  Feature extraction time: {feature_time:.2f}s\n")


# ================================================================
# STEP 3: CONSTRAINED OPTIMIZATION
# ================================================================
print("[STEP 3] Constrained Optimization")
print("-" * 70)

optimization_start = time.time()

optimal_config = solve_optimal_architecture(
    block_features,
    block_importance,
    max_gain,
    warmstart_acc,
    model,
    fullft_test_acc,
    fullft_time,
    CONSTRAINT_TYPE,
    TARGET_SPEEDUP if CONSTRAINT_TYPE == "speedup" else TARGET_PARAMS_PCT,
)

optimization_time = time.time() - optimization_start
print(f"  Optimization time: {optimization_time:.3f}s\n")


# ================================================================
# STEP 4: APPLY OPTIMAL ARCHITECTURE
# ================================================================
print("[STEP 4] Applying Optimal Architecture")
print("-" * 70)

apply_start = time.time()
blocks = get_block_dict(model)

for block_name in optimal_config["lora_blocks"]:
    block = blocks[block_name]

    for param in block.parameters():
        param.requires_grad = False

    if hasattr(block, "conv2") and not isinstance(block.conv2, LoRAConv2d):
        block.conv2 = LoRAConv2d(
            block.conv2,
            rank=optimal_config["lora_rank"],
            alpha=LORA_ALPHA,
        )

    print(f"  {block_name}: frozen + LoRA rank-{optimal_config['lora_rank']}")

for block_name in optimal_config["kept_blocks"]:
    block = blocks[block_name]
    for param in block.parameters():
        param.requires_grad = True
    print(f"  {block_name}: kept trainable")

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

apply_time = time.time() - apply_start

total_params, trainable_params, _ = count_params(model)
print("\n[Architecture Summary]")
print(f"  Trainable: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"  Reduction: {total_params / trainable_params:.1f}x")
print(f"  Application time: {apply_time:.2f}s\n")


# ================================================================
# STEP 5: FINE-TUNING
# ================================================================
print(f"[STEP 5] Fine-Tuning ({EPOCHS_FINETUNE} epochs)")
print("-" * 70)

finetune_start = time.time()
power_log = {"energy_J": 0.0, "last_t": finetune_start}

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable, lr=MAX_LR * 0.3, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer,
    max_lr=MAX_LR * 0.3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS_FINETUNE,
    pct_start=0.3,
    div_factor=10,
    final_div_factor=100,
)

best_val_acc = warmstart_acc
best_state = None

for epoch in range(1, EPOCHS_FINETUNE + 1):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device, power_log=power_log
    )
    val_loss, val_acc = evaluate(model, val_loader, device)
    print(
        f"  Epoch {epoch}/{EPOCHS_FINETUNE}: "
        f"train={train_acc:.2f}% val={val_acc:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)

finetune_time = time.time() - finetune_start
finetune_energy = power_log["energy_J"]

print(f"\nFinal val: {best_val_acc:.2f}%")
print(f"  Fine-tuning time: {finetune_time:.1f}s")
print(f"  Fine-tuning energy: {finetune_energy / 1000:.2f}kJ\n")


# ================================================================
# TOTAL TIME
# ================================================================
cdwf_total_time = time.time() - cdwf_total_start
cdwf_total_energy = warmstart_energy + finetune_energy

print("[TIMING AND ENERGY BREAKDOWN]")
print(
    f"  Warm-start:    {warmstart_time:6.1f}s "
    f"({100 * warmstart_time / cdwf_total_time:5.1f}%)  "
    f"Energy: {warmstart_energy / 1000:5.2f}kJ"
)
print(
    f"  Features:      {feature_time:6.1f}s "
    f"({100 * feature_time / cdwf_total_time:5.1f}%)"
)
print(
    f"  Optimization:  {optimization_time:6.1f}s "
    f"({100 * optimization_time / cdwf_total_time:5.1f}%)"
)
print(
    f"  Apply arch:    {apply_time:6.1f}s "
    f"({100 * apply_time / cdwf_total_time:5.1f}%)"
)
print(
    f"  Fine-tuning:   {finetune_time:6.1f}s "
    f"({100 * finetune_time / cdwf_total_time:5.1f}%)  "
    f"Energy: {finetune_energy / 1000:5.2f}kJ"
)
print("  ----------------------------------------")
print(
    f"  TOTAL CDWF:    {cdwf_total_time:6.1f}s (100.0%)  "
    f"Energy: {cdwf_total_energy / 1000:5.2f}kJ\n"
)


# ================================================================
# FINAL EVALUATION
# ================================================================
print("=" * 70)
print("  FINAL RESULTS")
print("=" * 70 + "\n")

_, test_acc = evaluate(model, test_loader, device)

retention = test_acc / fullft_test_acc
param_reduction = total_params / trainable_params
time_speedup = fullft_time / cdwf_total_time
energy_reduction = fullft_energy / cdwf_total_energy

print(f"Predicted Accuracy: {optimal_config['predicted_acc']:.2f}%")
print(f"Actual Test Accuracy: {test_acc:.2f}%")
print(f"Prediction Error: {abs(test_acc - optimal_config['predicted_acc']):.2f}%\n")

print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Retention: {retention * 100:.1f}%")
print(f"Trainable: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Parameter Reduction: {param_reduction:.1f}x\n")

print(f"Training Time: {cdwf_total_time:.1f}s")
print(f"Time Speedup: {time_speedup:.2f}x")
print(f"Target Speedup: {TARGET_SPEEDUP:.2f}x")
print(f"Constraint Met: {'Yes' if time_speedup >= TARGET_SPEEDUP * 0.9 else 'No'}\n")

print(f"Energy Reduction (measured): {energy_reduction:.2f}x")
print(f"  Full-FT energy: {fullft_energy / 1000:.2f}kJ")
print(f"  CDWF energy: {cdwf_total_energy / 1000:.2f}kJ\n")

print("=" * 70)
print("  COMPARISON: CIFAR-100 FULL_FT VS CDWF")
print("=" * 70)
print("Performance:")
print(f"  Full_FT: {fullft_test_acc:.2f}%")
print(f"  CDWF:    {test_acc:.2f}% ({retention * 100:.1f}% retention)")
print("\nEfficiency:")
print(f"  Parameters: {param_reduction:.1f}x fewer")
print(f"  Time:       {time_speedup:.2f}x faster (target: {TARGET_SPEEDUP:.2f}x)")
print(f"  Energy:     {energy_reduction:.2f}x less")
print("\nPrediction:")
print(f"  Predicted:  {optimal_config['predicted_acc']:.2f}%")
print(f"  Actual:     {test_acc:.2f}%")
print(f"  Error:      {abs(test_acc - optimal_config['predicted_acc']):.2f}%")
print("=" * 70 + "\n")


# ================================================================
# SAVE RESULTS
# ================================================================
results = {
    "method": "CDWF",
    "dataset": "cifar100",
    "constraint_type": CONSTRAINT_TYPE,
    "target_value": TARGET_SPEEDUP,
    "fullft_test_acc": fullft_test_acc,
    "test_acc": test_acc,
    "retention": retention,
    "trainable_params": trainable_params,
    "trainable_percent": 100 * trainable_params / total_params,
    "param_reduction": param_reduction,
    "training_time_s": cdwf_total_time,
    "time_speedup": time_speedup,
    "warmstart_energy_J": warmstart_energy,
    "finetune_energy_J": finetune_energy,
    "total_energy_J": cdwf_total_energy,
    "energy_reduction": energy_reduction,
    "predicted_acc": optimal_config["predicted_acc"],
    "prediction_error": abs(test_acc - optimal_config["predicted_acc"]),
    "n_kept_blocks": len(optimal_config["kept_blocks"]),
    "n_lora_blocks": len(optimal_config["lora_blocks"]),
    "lora_rank": optimal_config["lora_rank"],
}

df = pd.DataFrame([results])
if os.path.exists(RESULT_CSV):
    df = pd.concat([pd.read_csv(RESULT_CSV), df], ignore_index=True)
df.to_csv(RESULT_CSV, index=False)

print(f"Results saved to {RESULT_CSV}\n")

print("=" * 70)
print("  CDWF - CIFAR-100 COMPLETE")
print("=" * 70)
print(f"Retention: {retention * 100:.1f}%")
print(f"Speedup: {time_speedup:.2f}x")
print("=" * 70)